In [2]:
# Import required libraries

import xml.etree.ElementTree as ET
import pandas as pd

In [3]:
# Read XML file

tree = ET.parse("external_reviews.xml")
root = tree.getroot()

# Display root tag
root.tag

'reviews'

In [4]:
# Extract data from XML file

xml_data = []

for review in root.findall("review"):

    customer_id = review.find("customer_id").text
    rating = review.find("rating").text
    comments = review.find("comments").text

    xml_data.append({
        "customer_id": customer_id,
        "rating": rating,
        "comments": comments
    })

# Display extracted XML data

xml_data

[{'customer_id': '1', 'rating': '5', 'comments': 'Excellent service!'},
 {'customer_id': '2', 'rating': '4', 'comments': 'Good experience overall.'},
 {'customer_id': '3', 'rating': '2', 'comments': 'Late delivery.'}]

In [5]:
# Convert XML data into a DataFrame

xml_df = pd.DataFrame(xml_data)

# Display DataFrame

xml_df

,customer_id,rating,comments
0,1,5,Excellent service!
1,2,4,Good experience overall.
2,3,2,Late delivery.


In [6]:
# Check for missing values in XML data

xml_df.isnull().sum()

customer_id    0
rating         0
comments       0
dtype: int64

In [7]:
# Validate rating values in XML data

invalid_xml_ratings = xml_df[
    (xml_df["rating"].astype(int) < 1) |
    (xml_df["rating"].astype(int) > 5)
]

invalid_xml_ratings

,customer_id,rating,comments


In [8]:
# Check for duplicate records in XML data

duplicate_xml_records = xml_df[
    xml_df.duplicated()
]

duplicate_xml_records

,customer_id,rating,comments


In [9]:
# Add review date column to XML data

xml_df["review_date"] = "2024-01-01"

# Display updated DataFrame

xml_df

,customer_id,rating,comments,review_date
0,1,5,Excellent service!,2024-01-01
1,2,4,Good experience overall.,2024-01-01
2,3,2,Late delivery.,2024-01-01


In [10]:
# Import MySQL connector

import mysql.connector

# Connect to MySQL database

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="55003698",
    database="etl_final_project"
)

# Create cursor object

cursor = connection.cursor()

print("Connected to MySQL successfully!")

Connected to MySQL successfully!


In [11]:
# Insert XML data into external_reviews_staging table

insert_query = """
INSERT INTO external_reviews_staging
(customer_id, rating, comments, review_date)
VALUES (%s, %s, %s, %s)
"""

# Insert rows into table

for index, row in xml_df.iterrows():
    cursor.execute(insert_query, (
        row["customer_id"],
        row["rating"],
        row["comments"],
        row["review_date"]
    ))

# Commit changes

connection.commit()

print("XML data inserted successfully!")

XML data inserted successfully!


In [12]:
# Verify inserted XML data

cursor.execute("SELECT * FROM external_reviews_staging")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 1, 5, 'Excellent service!', datetime.date(2024, 1, 1))
(2, 2, 4, 'Good experience overall.', datetime.date(2024, 1, 1))
(3, 3, 2, 'Late delivery.', datetime.date(2024, 1, 1))


In [15]:
# Check Reviews table structure
cursor.execute("DESCRIBE Reviews")
for row in cursor.fetchall():
    print(row)

print("\n--- external_reviews_staging structure ---")

# Check external_reviews_staging table structure
cursor.execute("DESCRIBE external_reviews_staging")
for row in cursor.fetchall():
    print(row)

('review_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('customer_id', 'int', 'NO', 'MUL', None, '')
('product_id', 'int', 'NO', 'MUL', None, '')
('review_date', 'date', 'YES', 'MUL', None, '')
('rating', 'int', 'NO', '', None, '')
('comments', 'text', 'YES', '', None, '')

--- external_reviews_staging structure ---
('review_id', 'int', 'NO', 'PRI', None, 'auto_increment')
('customer_id', 'int', 'YES', '', None, '')
('rating', 'int', 'YES', '', None, '')
('comments', 'text', 'YES', '', None, '')
('review_date', 'date', 'YES', '', None, '')


# Step 30 completed successfully
# The validated XML review records were transferred from the 
# external_reviews_staging table into the final Reviews table 
# in MySQL. Duplicate prevention logic was applied using 
# a WHERE NOT EXISTS condition to maintain data integrity. 
# The final verification confirmed that the XML review data 
# was successfully integrated into the main reviews dataset.